In [48]:
print("Hello")

Hello


In [49]:
%pwd

'c:\\Users\\Admin\\Documents\\PJMedicalChat'

In [50]:
import os
os.chdir("../")

In [51]:
%pwd

'c:\\Users\\Admin\\Documents'

In [52]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [53]:
#Extract Data From the PDF File
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents


In [54]:
%pwd

'c:\\Users\\Admin\\Documents'

In [57]:
cd PJMedicalChat/MedicalChatBot/

c:\Users\Admin\Documents\PJMedicalChat\MedicalChatBot


c:\Users\Admin\miniconda3\envs\medibot_310\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [58]:
extracted_data=load_pdf_file(data='Data/')

In [59]:
extracted_data[0].page_content[:1000]

''

In [60]:
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks


In [61]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))


Length of Text Chunks 5860


In [62]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))


Length of Text Chunks 5860


In [63]:
#text_chunks

In [64]:
from langchain.embeddings import HuggingFaceEmbeddings

In [65]:
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings


In [66]:
pip install sentence-transformers


Note: you may need to restart the kernel to use updated packages.


In [67]:
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [68]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [69]:
embeddings = download_hugging_face_embeddings()

In [70]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [71]:
from dotenv import load_dotenv
load_dotenv()


True

In [72]:
%pwd

'c:\\Users\\Admin\\Documents\\PJMedicalChat\\MedicalChatBot'

In [73]:
import os
os.chdir("../")

In [74]:
%pwd

'c:\\Users\\Admin\\Documents\\PJMedicalChat'

In [75]:
from dotenv import load_dotenv
from pathlib import Path

dotenv_path = Path('./MedicalChatBot/.env')  # Thay đổi đường dẫn nếu cần
load_dotenv(dotenv_path=dotenv_path)

True

In [76]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

In [77]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [78]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)
index_name = "test1"

In [79]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name= "test1",
    embedding=embeddings, 
)


In [80]:

# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name="test1",
    embedding=embeddings
)


In [81]:
docsearch

In [82]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [83]:

retrieved_docs = retriever.invoke("What is Acne?")

In [84]:

retrieved_docs

[Document(id='322c0045-1907-4ee7-ba24-676a231ae5a7', metadata={'page': 39.0, 'source': 'Data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='08bc4ce4-250a-4401-a7c6-deb5a68385ca', metadata={'page': 39.0, 'source': 'Data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='b21be003-1b49-422e-843f-3d56e4a88f9c', metadata={'page': 39.0, 'source': 'Data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26')]

In [85]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

In [86]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


In [87]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.4, max_output_tokens=500, google_api_key=GEMINI_API_KEY)

In [88]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "i got headeache?"})
print(response["answer"])

I'm sorry, but the provided context does not contain information about headaches. It only contains information about alopecia. Therefore, I cannot answer your question.
